# Video Transcriber

Paste a video link (Instagram, YouTube, TikTok, etc.) and get the full spoken transcript.

**Requirements:** `pip install -r requirements.txt` + [ffmpeg](https://ffmpeg.org/download.html) installed on PATH

In [ ]:
# Install dependencies (run once)
# !pip install yt-dlp openai-whisper

In [ ]:
import subprocess
import tempfile
import os
import whisper
from pathlib import Path

In [ ]:
# Load Whisper model (options: tiny, base, small, medium, large)
# 'base' is fast and decent quality. Use 'medium' or 'large' for better accuracy.
model = whisper.load_model("base")

In [ ]:
def transcribe_video(url: str, model_size: str = None) -> dict:
    """
    Download audio from a video URL and transcribe it.
    
    Args:
        url: Any video URL supported by yt-dlp (Instagram, YouTube, TikTok, etc.)
        model_size: Override whisper model (tiny/base/small/medium/large)
    
    Returns:
        dict with 'text' (full transcript) and 'segments' (timestamped chunks)
    """
    global model
    
    if model_size:
        model = whisper.load_model(model_size)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        audio_path = os.path.join(tmpdir, "audio.mp3")
        
        # Download audio only
        cmd = [
            "yt-dlp",
            "--extract-audio",
            "--audio-format", "mp3",
            "--audio-quality", "0",
            "-o", audio_path,
            url
        ]
        
        print(f"Downloading audio from: {url}")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"Error downloading: {result.stderr}")
            return None
        
        # Find the downloaded file (yt-dlp may adjust extension)
        audio_file = None
        for f in os.listdir(tmpdir):
            if f.startswith("audio"):
                audio_file = os.path.join(tmpdir, f)
                break
        
        if not audio_file:
            print("Error: No audio file downloaded")
            return None
        
        print("Transcribing...")
        result = model.transcribe(audio_file)
        
        return result

In [ ]:
# === PASTE YOUR VIDEO LINK HERE ===
url = "https://www.instagram.com/reel/DZvAMYZA1ae/?igsh=MWpiMHF2ZTVhejJjbA=="

result = transcribe_video(url)

if result:
    print("\n" + "="*60)
    print("TRANSCRIPT")
    print("="*60)
    print(result["text"])

In [ ]:
# View timestamped segments
if result:
    print("\nTimestamped segments:\n")
    for seg in result["segments"]:
        start = f"{seg['start']:.1f}s"
        end = f"{seg['end']:.1f}s"
        print(f"[{start} -> {end}] {seg['text']}")